In [1]:
!pip install -q easyocr fastapi uvicorn nest-asyncio

# Download cloudflared
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

import nest_asyncio
nest_asyncio.apply()

print("Environment ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 31.5 MB/s eta 0:00:00
Environment ready.


In [5]:
import easyocr
import torch
import time
import numpy as np

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\nLoading EasyOCR...")
start = time.time()

reader = easyocr.Reader(lang_list=["ar","fa","ur","ug","en"], gpu=torch.cuda.is_available())

print(f"Ready in {time.time() - start:.1f}s")

# Warmup
_ = reader.readtext(np.zeros((100, 100, 3), dtype=np.uint8))
print("Warmup done.")

CUDA: True
GPU: Tesla T4

Loading EasyOCR...
Ready in 5.8s
Warmup done.


In [6]:
import numpy as np
from PIL import Image
import io
import base64
import re
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional

In [7]:
def correct_ocr_text(text):
    """Fix common Arabic OCR errors."""
    if not text:
        return ""
    text = re.sub(r'[\u200b-\u200f\u202a-\u202e\u2060-\u2069]', '', text)
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ـ+', '', text)
    text = re.sub(r'\s+', ' ', text)
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        s = line.strip()
        if not s:
            continue
        meaningful = sum(1 for c in s if c.isalpha())
        if len(s) > 0 and meaningful / len(s) < 0.2:
            continue
        cleaned.append(line)
    return '\n'.join(cleaned).strip()

In [8]:
app = FastAPI(title="Moroccan Legal RAG - EasyOCR Service", version="1.0.0")

class OCRRequest(BaseModel):
    """Base64-encoded image for OCR."""
    image_base64: str
    correct_text: bool = True

class OCRLine(BaseModel):
    text: str
    confidence: float
    bbox: List[List[float]]

class OCRResponse(BaseModel):
    text: str
    lines: List[OCRLine]
    total_chars: int
    arabic_chars: int
    french_chars: int
    confidence: float
    time_ms: float

@app.get("/")
async def root():
    return {
        "service": "Moroccan Legal RAG - EasyOCR Service",
        "engine": "EasyOCR",
        "languages": ["ar", "fr", "en"],
        "gpu": torch.cuda.is_available(),
        "endpoints": ["/health", "/ocr", "/docs"],
    }

@app.get("/health")
async def health():
    return {
        "status": "ok",
        "engine": "EasyOCR",
        "languages": ["ar", "fr", "en"],
        "gpu": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    }

@app.post("/ocr", response_model=OCRResponse)
async def ocr(request: OCRRequest):
    # Decode image
    img_bytes = base64.b64decode(request.image_base64)
    img = Image.open(io.BytesIO(img_bytes))
    img_array = np.array(img)

    # Run OCR
    start = time.time()
    raw_results = reader.readtext(img_array)
    elapsed = (time.time() - start) * 1000

    # Process results
    lines = []
    texts = []
    confidences = []

    for item in raw_results:
        bbox, text, conf = item
        corrected = correct_ocr_text(text) if request.correct_text else text

        lines.append(OCRLine(
            text=corrected,
            confidence=round(conf * 100, 1),
            bbox=[[round(p[0], 1), round(p[1], 1)] for p in bbox],
        ))
        texts.append(corrected)
        confidences.append(conf)

    full_text = "\n".join(texts)
    avg_conf = (sum(confidences) / len(confidences) * 100) if confidences else 0

    # Character counts
    arabic = sum(1 for c in full_text if '\u0600' <= c <= '\u06ff')
    french = sum(1 for c in full_text if c.isascii() and c.isalpha())

    return OCRResponse(
        text=full_text,
        lines=lines,
        total_chars=len(full_text),
        arabic_chars=arabic,
        french_chars=french,
        confidence=round(avg_conf, 1),
        time_ms=round(elapsed, 2),
    )

print("FastAPI app ready.")

FastAPI app ready.


In [ ]:
import subprocess
import threading
import uvicorn
import re

def start_tunnel():
    time.sleep(3)

    process = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    for line in process.stdout:
        print(line.strip())
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if match:
            print()
            print("=" * 60)
            print("PUBLIC TUNNEL URL:")
            print(f"   {match.group(0)}")
            print("=" * 60)
            print()
            print("Add to your local .env:")
            print(f'  EASYOCR_SERVICE_URL={match.group(0)}')
            print()
            print("Test it:")
            print(f"  curl {match.group(0)}/health")
            print()

tunnel_thread = threading.Thread(target=start_tunnel, daemon=True)
tunnel_thread.start()

print("Starting EasyOCR service...")
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [2796]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Starting EasyOCR service...
2026-05-25T16:57:12Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-25T16:57:12Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-25T16:57:16Z INF +--------------------------------------------------------------------------------------------+
2026-05-25T16:57:16Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-25T16:57:16Z INF |  https://train-vienna-spor